# EDS contact-topology analysis — step-by-step walkthrough

This notebook teaches the analysis used for the CNT/PVDF-SDC vs cast CNT/PVDF comparison, one step at a time, and ends by exporting every result to Excel/CSV, TIFF and editable figures.

**How to run**

* **Anaconda (Jupyter):** put this notebook, `eds_topology.py`, `samples.csv` and a folder `reports/` containing the AZtec `.docx` site reports in the same folder. Open the notebook and run each cell with **Shift + Enter**.
* **Google Colab (no installation):** open the notebook in Colab, run the *Colab only* cell below, and upload `eds_topology.py`, `samples.csv` and the `.docx` reports when asked.

Each step has a short explanation, the code, and a **Check** that tells you what you should see. If what you see is different, stop and look before moving on.

### 0. Setup

In [1]:
# Colab only: upload eds_topology.py, samples.csv and the AZtec .docx reports. Skip this cell in Anaconda.
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import files
    os.makedirs('reports', exist_ok=True)
    for name in files.upload():
        if name.lower().endswith('.docx'):
            os.replace(name, os.path.join('reports', name))
    print('reports/:', os.listdir('reports'))

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import eds_topology as et          # the analysis functions (eds_topology.py must be in this folder)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
REPORTS = 'reports'                 # folder with the AZtec .docx files
print(sorted(os.listdir(REPORTS)))

ModuleNotFoundError: No module named 'numpy'

### 1. Look inside one AZtec report

An AZtec *Site Report* (.docx) stores pictures in this order: the **layered image** first, then **one picture per element map**, then the **map-sum spectrum**. The report does not store the element names with the pictures, so you assign them yourself by looking.

**Check:** the C map (usually red) shows both the CNT clusters and the PVDF; the F map (usually green) shows only the PVDF fibrils.

In [ ]:
path = os.path.join(REPORTS, 'CNt-PVDF_sdc film 2_2026-10-06_18-41-35.docx')   # change to any report
imgs = et.read_aztec_report(path)
fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 2.6))
for i, (ax, im) in enumerate(zip(axes, imgs)):
    ax.imshow(im); ax.set_axis_off()
    ax.set_title('layered image' if i == 0 else ('spectrum' if i == len(imgs) - 1 else f'map {i}'), fontsize=9)
plt.show()

### 2. Load the maps as numbers

`load_maps` turns each map picture into a grid of numbers (brightness = X-ray counts, up to a display scaling), removes the black frame, and averages the up-scaled report picture back to the real map grid (512 pixels wide).

Give the element of each map **in the order you saw in step 1**. Use `'X'` for a map you want to ignore (for example the Re/Os/Pt maps that AZtec mis-assigns on a Si wafer).

In [ ]:
ELEMENTS = ['C', 'F', 'O']          # order of 'map 1, map 2, map 3' above
maps = et.load_maps(path, ELEMENTS)
for k, v in maps.items():
    print(k, v.shape, f'min {v.min():.0f}  median {np.median(v):.0f}  max {v.max():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(maps['C'], cmap='Reds_r'); axes[0].set_title('C map')
axes[1].imshow(maps['F'], cmap='Greens_r'); axes[1].set_title('F map')
for ax in axes: ax.set_axis_off()
plt.show()

### 3. The key idea: the C–F triangle

PVDF is (CH₂–CF₂)ₙ, so it gives **both** C and F. CNTs give **only** C. Pores, shadows (or bare Si) give **neither**.

Plot every pixel by its C signal (x-axis) and F signal (y-axis). The cloud fills a triangle:

```
 F ^          PVDF corner (C + F)
   |          *
   |         / \
   |        /   \      a pixel's position = its mix
   |       /     \
   |      *-------*
   |  background   CNT corner (C, no F)
   +---------------------> C
```

The maps are first smoothed slightly (Gaussian, σ = 1 pixel) to reduce counting noise.

**Check:** you should see a triangular cloud. For a shear-assembled SDC film, the pixels crowd toward the CNT and PVDF corners. For a cast film, they lie along the middle, between background and the C+F direction, because the polymer is mixed through the CNTs.

In [ ]:
Cs = ndi.gaussian_filter(maps['C'], 1.0)
Fs = ndi.gaussian_filter(maps['F'], 1.0)
from matplotlib.colors import LogNorm
plt.figure(figsize=(4, 3.4))
plt.hist2d(Cs.ravel(), Fs.ravel(), bins=120, cmap='Greys', norm=LogNorm())
plt.xlabel('C signal'); plt.ylabel('F signal'); plt.title('Every pixel of the map')
plt.show()

**Try this:** before running the next cell, point on the plot to where you think the three corners are.

### 4. Find the three corners automatically

* **Background corner** (films): the median of the darkest 3 % of pixels (lowest C + F). For particles on a Si wafer, use the brightest 30 % of the Si map instead (`background='si'`).
* **CNT and PVDF corners:** seen from the background corner, the cloud spans a fan of directions. The CNT corner lies along the lowest-angle edge (most C, least F) and the PVDF corner along the highest-angle edge (most F). Each corner is placed at the 97th percentile of distance along that edge, so a few hot pixels cannot drag it.

**Check:** the red triangle should enclose almost all of the cloud.

In [ ]:
S, CNT, PVDF = et.find_corners(Cs, Fs, background='low')
print('background corner (C, F):', S.round(1))
print('CNT corner        (C, F):', CNT.round(1))
print('PVDF corner       (C, F):', PVDF.round(1))

plt.figure(figsize=(4, 3.4))
plt.hist2d(Cs.ravel(), Fs.ravel(), bins=120, cmap='Greys', norm=LogNorm())
tri = np.array([S, CNT, PVDF, S])
plt.plot(tri[:, 0], tri[:, 1], 'r-o')
for p_, lab in [(S, 'background'), (CNT, 'CNT'), (PVDF, 'PVDF')]:
    plt.annotate(lab, p_, xytext=(5, 5), textcoords='offset points', color='r')
plt.xlabel('C signal'); plt.ylabel('F signal'); plt.show()

### 5. Unmix one pixel by hand

Any point **p** inside the triangle can be written as

  **p** = **S** + a·(**CNT** − **S**) + b·(**PVDF** − **S**)

where *a* is the CNT abundance and *b* the PVDF abundance (both between 0 and 1). That is two equations (one for C, one for F) with two unknowns, so it can be solved exactly. The CNT fraction of the material in that pixel is

  **x = a / (a + b)**  → x = 1 pure CNT, x = 0 pure PVDF.

Two details: negative abundances (pixels just outside an edge, from noise) are set to 0, and if a + b > 1 (the pixel lies beyond the CNT–PVDF edge) both are scaled down so that a + b = 1. Neither changes x much.

**Check:** after the same clipping and rescaling, the hand calculation and `et.unmix` give identical a, b and x.

In [ ]:
row, col = 200, 250                       # pick any pixel
p = np.array([Cs[row, col], Fs[row, col]])
T = np.column_stack([CNT - S, PVDF - S])  # 2 x 2 matrix
a, b = np.linalg.solve(T, p - S)
print(f'pixel ({row},{col}): C = {p[0]:.1f}, F = {p[1]:.1f}')
print(f'solve:     a = {a:.3f}, b = {b:.3f}   (a + b = {a + b:.3f})')
a, b = max(a, 0), max(b, 0)                 # clip negatives
scale = max(a + b, 1)                       # rescale if beyond the CNT–PVDF edge
a, b = a / scale, b / scale
print(f'by hand:   CNT abundance a = {a:.3f}, PVDF abundance b = {b:.3f}, x = {a / (a + b):.3f}')

fc, fp = et.unmix(Cs, Fs, S, CNT, PVDF)
print(f'et.unmix:  a = {fc[row, col]:.3f}, b = {fp[row, col]:.3f}, x = {fc[row, col] / (fc[row, col] + fp[row, col]):.3f}')

### 6. Classify the whole map

1. **Material mask:** pixels whose total abundance (a + b) is high enough (Otsu threshold with hysteresis); the rest is *low signal* (pore or shadow). Specks of ≤ 12 pixels are removed.
2. **Classes:** CNT-rich if x > 0.6, PVDF-rich if x < 0.4, mixed in between. CNT specks of ≤ 4 pixels are removed.

**Check:** CNT-rich (orange) should match the red-only regions of the C/F overlay, and PVDF-rich (green) the fibrils.

In [ ]:
g = et.segment(maps, background='low')     # default settings
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].imshow(et.cf_overlay(maps)); axes[0].set_title('C (red) / F (green)')
im = axes[1].imshow(np.where(g['M'], g['x'], np.nan), cmap='RdYlBu_r', vmin=0, vmax=1); axes[1].set_title('CNT fraction x')
plt.colorbar(im, ax=axes[1], fraction=0.035)
axes[2].imshow(et.class_rgb(g['labels']), interpolation='nearest'); axes[2].set_title('classes')
for ax in axes: ax.set_axis_off()
plt.show()
n = g['M'].sum()
print(f"CNT-rich {g['CNT'].sum() / n:.0%}   mixed {g['MIX'].sum() / n:.0%}   PVDF-rich {g['PV'].sum() / n:.0%}   low signal {1 - g['M'].mean():.0%} of field")

### 7. Topology metrics

| Metric | How it is computed | How to read it |
|---|---|---|
| **C–F correlation** | Pearson r of smoothed C and F over material pixels | Negative: CNT and PVDF sit in separate regions. Positive: polymer mixed through the CNTs |
| **F in CNT-rich domains** | F of the CNT corner above background, divided by F of the PVDF corner above background | ≤ 0: CNT domains carry no polymer signal (exposed). > 0: polymer present in CNT domains |
| **Low-signal area** | fraction of the field outside the material mask | open pores / shadows |
| **PVDF in largest network** | share of PVDF-rich pixels in the biggest connected PVDF region | continuity of the polymer network |
| **PVDF within 5 px of CNT** | share of PVDF-rich pixels whose path distance (inside the material) to a CNT-rich pixel is ≤ 5 px | proximity; high values can also mean polymer blended *inside* CNT clumps |

**Check:** for an SDC film you expect r < 0 and F in CNT ≤ 0; for a cast film r > 0 and F in CNT > 0.

In [ ]:
m, dist = et.metrics(g)
m['F_in_CNT'] = et.F_in_CNT(maps)[0]
pd.Series({et.METRIC_INFO[k][0]: v for k, v in m.items()}).round(3)

### 8. Is the result robust? (sensitivity check)

Every analysis has choices (smoothing, thresholds, corner positions). `analyze_sample` repeats steps 4–7 under **10 settings** and reports the baseline value plus the minimum and maximum. A conclusion is robust when the ranges of the two sample types do not overlap.

In [ ]:
res = et.analyze_sample(path, ELEMENTS, sample='film 2', group='SDC')
pd.DataFrame({et.METRIC_INFO[k][0]: v for k, v in res['metrics'].items()}, index=['baseline', 'min', 'max']).T.round(3)

In [ ]:
pd.DataFrame([{'setting': i, **{**et.DEFAULTS, **s}} for i, s in enumerate(et.SETTINGS)])   # the 10 settings

### 9. Analyse all samples

Edit `samples.csv` (open it in Excel) to list your reports. One row per map:

| column | meaning |
|---|---|
| `file` | file name of the AZtec report in `reports/` |
| `sample` | short name used in tables and figures |
| `group` | e.g. `SDC`, `Cast`, `Post-mixed` |
| `elements` | element of each map in report order, separated by `;` (e.g. `C;F;O`) |
| `background` | `low` for films, `si` for particles on a Si wafer |

Each sample takes about 10 s (the 10 settings).

In [ ]:
table = pd.read_csv('samples.csv').fillna('')
display(table)
results = et.analyze_batch(table.to_dict('records'), folder=REPORTS)

In [ ]:
stats = et.group_stats(results)
stats.round(4)

**Reading the statistics.** `Welch t-test p` compares the group means (n per group shown). With 3 vs 3 samples the exact Mann–Whitney p cannot go below 0.10, whatever the data, so report mean ± SD (and the ranges) and name the test and n if you quote p. `groups overlap across settings?` = *no* means every sample of one group stays separated from every sample of the other under all 10 settings.

### 10. Export everything

`export_all` writes a folder you can open in Excel, Origin, PowerPoint and ImageJ:

| Output | Use |
|---|---|
| `EDS_topology_results.xlsx` | all tables, one sheet each: Summary, Ranges, Group_stats, x_distribution, Distance_CDF, Corners, Metric_definitions, Settings |
| `csv/` | the same tables as CSV (File → Import in Origin) |
| `raw/<sample>_*.csv` | full pixel matrices: C and F maps (raw and smoothed), CNT fraction, class labels (0 low signal, 1 PVDF-rich, 2 mixed, 3 CNT-rich). Open directly in Excel |
| `data/<sample>_scatter.csv` | 5000 random pixels (C, F, class) to re-plot the triangle in Origin |
| `maps/` | class maps (PNG for slides; TIFF labels and 32-bit CNT-fraction for ImageJ) |
| `figures/` | class maps, triangles, metric dot-plot and distance curves as **SVG/PDF** (editable) and PNG |

To edit an SVG in PowerPoint: *Insert → Pictures → This Device*, select the `.svg`, then right-click → *Convert to Shape*. Text stays editable. Illustrator and Inkscape open the SVG/PDF directly.

In [ ]:
OUT = 'EDS_topology_results'
et.export_all(results, OUT)
if 'google.colab' in sys.modules:          # Colab: zip and download the folder
    import shutil
    shutil.make_archive(OUT, 'zip', OUT)
    files.download(OUT + '.zip')

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(os.path.join(OUT, 'figures', 'metrics_dotplot.png'), width=900))

### Checklist for new samples (cast replicates, post-mixed film, second map per sample)

1. Acquire every map with the **same mode, kV, magnification (field width), pixel grid, dwell time and frames**. Maps taken in a different mode are not comparable (this is why *Site 3* was excluded).
2. Record the **field width** so pixels can be converted to µm (µm per pixel = field width / 512).
3. Remove mis-identified elements in AZtec (e.g. Re, Os, Pt on Si) before exporting the report, or mark those maps `X`.
4. Prefer **≥ 2 maps per sample** and ≥ 3 samples per group.
5. Look at the triangle (step 3–4) for every new map before trusting its numbers.